In [9]:
from rdflib import Graph, plugin, URIRef, Literal
from rdflib.serializer import Serializer

import re

import json
import requests

import requests_cache

requests_cache.install_cache('plod_cache',backend='memory', expire_after=300)

In [10]:
url = requests.get("https://p-lod.umasscreate.net/api/items?item_set_=2&per_page=10")
jsonitems = re.sub(r'"https:\/\/p-lod.umasscreate.net\/api-context"',
r'{"o":"http:\/\/omeka.org\/s\/vocabs\/o#","dcterms":{"@id":"http:\/\/purl.org\/dc\/terms\/"},"dctype":{"@id":"http:\/\/purl.org\/dc\/dcmitype\/"},"bibo":{"@id":"http:\/\/purl.org\/ontology\/bibo\/"},"owl":{"@id":"http:\/\/www.w3.org\/2002\/07\/owl#"},"rdf":{"@id":"http:\/\/www.w3.org\/1999\/02\/22-rdf-syntax-ns#"},"rdfs":{"@id":"http:\/\/www.w3.org\/2000\/01\/rdf-schema#"},"p-lod":{"@id":"http:\/\/p-lod.umasscreate.net\/vocabulary#"},"o-cnt":"http:\/\/www.w3.org\/2011\/content#","o-time":"http:\/\/www.w3.org\/2006\/time#"}',
      url.text)

url = requests.get("https://p-lod.umasscreate.net/api/items?item_set_id=1")
jsonclasses = re.sub(r'"https:\/\/p-lod.umasscreate.net\/api-context"',
r'{"o":"http:\/\/omeka.org\/s\/vocabs\/o#","dcterms":{"@id":"http:\/\/purl.org\/dc\/terms\/"},"dctype":{"@id":"http:\/\/purl.org\/dc\/dcmitype\/"},"bibo":{"@id":"http:\/\/purl.org\/ontology\/bibo\/"},"owl":{"@id":"http:\/\/www.w3.org\/2002\/07\/owl#"},"rdf":{"@id":"http:\/\/www.w3.org\/1999\/02\/22-rdf-syntax-ns#"},"rdfs":{"@id":"http:\/\/www.w3.org\/2000\/01\/rdf-schema#"},"p-lod":{"@id":"http:\/\/p-lod.umasscreate.net\/vocabulary#"},"o-cnt":"http:\/\/www.w3.org\/2011\/content#","o-time":"http:\/\/www.w3.org\/2006\/time#"}',
      url.text)


In [ ]:
# print(jsonitems)

In [11]:
g_classes = Graph().parse(data=jsonclasses, format='json-ld')

In [12]:
g_classes_use = g_classes
g_classes_use.parse(data=jsonitems, format='json-ld')

<Graph identifier=N53b9515199bf4749937c7cf6d2887a9d (<class 'rdflib.graph.Graph'>)>

In [ ]:
# print(g_classes_use.serialize(format='turtle').decode('utf-8'))

In [13]:
rdf_construct  = g_classes_use.query("""
CONSTRUCT { ?new_s a ?type ; ?p_local ?o_local .}
WHERE {?s <http://omeka.org/s/vocabs/o#item_set> <https://p-lod.umasscreate.net/api/item_sets/2> . 

       ?s a ?type_as_item .
       ?type_as_item <http://omeka.org/s/vocabs/o#item_set> <https://p-lod.umasscreate.net/api/item_sets/1> .
       ?type_as_item <http://purl.org/dc/terms/identifier> ?type_as_label .
       BIND(IRI(concat("https://p-lod.umasscreate.net/p-lod/id/",?type_as_label)) as ?type)
       
       ?s <http://purl.org/dc/terms/identifier> ?item_identifier .
       BIND(IRI(CONCAT("https://p-lod.umasscreate.net/p-lod/id/",?item_identifier)) as ?new_s)
       
       ?s ?p_local ?o_local .
       FILTER regex(str(?p_local), "http://p-lod.umasscreate.net/vocabulary#") . 
       
       }
       
""")


In [14]:
new_g = Graph()
new_g.namespace_manager.bind('p-lod-vocab', URIRef('http://p-lod.umasscreate.net/vocabulary#'))
new_g.namespace_manager.bind('p-lod', URIRef('https://p-lod.umasscreate.net/p-lod/id/'))
for triple in rdf_construct:
    if "https://p-lod.umasscreate.net/api/items/" in triple[2]:
        sub_item_json = requests.get(triple[2]).text
        new_uri = URIRef("https://p-lod.umasscreate.net/p-lod/id/" + json.loads(sub_item_json)['dcterms:identifier'][0]['@value'])
        new_g.add((triple[0],triple[1],new_uri))
    elif str(triple[1]) == "http://p-lod.umasscreate.net/vocabulary#area":
        new_g.add((triple[0],triple[1],Literal(float(triple[2]))))
    else:
        new_g.add(triple)

ttl = new_g.serialize(format="turtle").decode('utf-8')

In [15]:
ttl_file = open("p-lod.ttl", "w")
ttl_file.write(ttl)
ttl_file.close()